# CCA-F 考试复习 Notebook — Prompt Engineering

本 notebook 按你给定的 CCA-F task 列表重写。每个 Task 都包含概念解释、可运行 Python 示例、考点总结和 2 道模拟题。

## 环境初始化

In [ ]:
import json
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional

MODEL = "claude-haiku-4-5-20251001"
print("环境就绪：本 notebook 使用本地模拟，代码结构贴近 Anthropic Messages API / Claude Code 工作流。")

## D4.1 — 应用 Few-shot 技术与结构化 Prompt 模式

### 📖 概念解释

Few-shot 提示不是“多写几个例子”而已，而是用少量高质量样例给模型建立任务边界、输出格式和错误边界。CCA-F 常考的判断是：当任务存在分类边界、字段抽取歧义或固定格式要求时，2-4 个覆盖代表性场景的正例、反例和边界例，通常比更长的自然语言说明更有效。

结构化 prompt 应把角色、规则、示例、输入资料和输出契约分开。XML 标签常用于 Claude，因为它能清楚标出 `<role>`、`<rules>`、`<examples>`、`<document>`、`<output_format>` 等区块，避免模型把示例当成待处理数据。角色 assignment 要具体到能力和责任，例如“你是发票字段抽取器”，而不是泛泛地说“你很聪明”。

In [ ]:
# Task 4.1：Few-shot、正反例、XML 标签与角色设定
# 本例不调用模型，而是展示生产 prompt 的结构，并用本地函数模拟期望输出。

import json
import re

prompt = """<role>你是一个谨慎的发票字段抽取器，只输出符合 JSON 契约的结果。</role>
<rules>
- 只抽取 <document> 中明确出现的字段。
- 缺失字段必须返回 null；不要猜测或补全。
- 货币只能是 USD、CAD、EUR 或 null。
</rules>
<examples>
<example type="positive">
  <document>Invoice INV-100. Total: $42.00 USD</document>
  <answer>{"invoice_id":"INV-100","total":42.0,"currency":"USD"}</answer>
</example>
<example type="negative">
  <document>Invoice INV-101. Amount due: not provided.</document>
  <answer>{"invoice_id":"INV-101","total":null,"currency":null}</answer>
</example>
<example type="boundary">
  <document>Quote Q-1. Estimated total may be 80 CAD.</document>
  <answer>{"invoice_id":null,"total":null,"currency":"CAD"}</answer>
</example>
</examples>
<output_format>{"invoice_id": string|null, "total": number|null, "currency": "USD"|"CAD"|"EUR"|null}</output_format>
<document>Invoice INV-9, Total: $88.10 USD</document>
"""

# 简单模拟：真实系统会把 prompt 发送给模型；这里保证 notebook 可离线运行。
def extract_invoice_locally(document: str) -> dict:
    invoice_match = re.search(r"Invoice\s+(INV-\d+)", document)
    total_match = re.search(r"Total:\s*\$([0-9]+(?:\.[0-9]{2})?)", document)
    currency_match = re.search(r"\b(USD|CAD|EUR)\b", document)
    return {
        "invoice_id": invoice_match.group(1) if invoice_match else None,
        "total": float(total_match.group(1)) if total_match else None,
        "currency": currency_match.group(1) if currency_match else None,
    }

source_document = "Invoice INV-9, Total: $88.10 USD"
print("结构化 prompt 片段：")
print(prompt.split("<document>")[-1].strip())
print("本地模拟输出：")
print(json.dumps(extract_invoice_locally(source_document), ensure_ascii=False, indent=2))

### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 用少量高质量 few-shot 样例覆盖正常、反例和边界情况。
- 用 XML 标签分隔角色、规则、示例、资料和输出格式，降低指令与数据混淆。
- 角色 assignment 要具体说明任务责任、约束和输出目标。
- 负例应展示“不应该抽取/不应该猜测”的情况，而不是只展示失败文本。

**✗ 反模式**

- 用很长的说明替代样例，导致边界仍不清楚。
- 只给正例，不说明缺失、模糊、冲突信息如何处理。
- 要求模型展示隐藏推理链，而不是要求简洁理由、校验结果或结构化字段。
- 标签嵌套混乱，或把真实输入放进 examples，导致模型误把输入当示例。

### 🧪 模拟题

**Q1.** 一个 Claude 应用需要把客服请求分为 `refund`、`technical_issue`、`billing_question`。模型经常把“发票地址错误”误分到 `technical_issue`。最符合 CCA-F 的改进是？

A) 增加包含“发票地址错误 => billing_question”的 few-shot 边界例，并保留输出枚举  
B) 把 temperature 调高，让模型探索更多分类  
C) 删除分类枚举，让模型自由命名类别  
D) 要求模型输出完整隐藏推理链

> **答案：A**  
> **解析：** 边界错误优先用针对性 few-shot 样例和明确枚举修正；提高随机性或放开类别会降低稳定性。

**Q2.** 哪个 prompt 结构最适合降低“把示例当作真实输入处理”的风险？

A) 用 `<examples>` 和 `<document>` 明确分隔示例与当前输入  
B) 把示例、规则、输入全部放在同一段自然语言里  
C) 只写“你是专家，请严格遵守要求”  
D) 去掉负例，避免模型看到错误形式

> **答案：A**  
> **解析：** XML 标签能清楚隔离示例和当前资料；负例本身不是问题，混乱的边界才是问题。

## D4.2 — 使用 JSON Schema 与 Pydantic 强制结构化输出

### 📖 概念解释

结构化输出的核心不是“请输出 JSON”，而是把输出契约移到可验证的 schema 或类型模型中。JSON Schema / Pydantic 风格验证可以检查必填字段、类型、枚举、范围和跨字段业务规则。考试常见陷阱是把“JSON 可解析”误认为“结果正确”：合法 JSON 仍可能有错误字段类型、非法枚举或与源资料冲突的值。

生产级设计通常包含三层：prompt 中声明输出格式，模型调用或工具 schema 约束结构，程序化验证器检查结果。验证失败时，把具体错误反馈给模型进行 self-correction；如果源资料缺失，应允许 `null` 或 `unknown`，而不是迫使模型编造。

In [ ]:
# Task 4.2：JSON Schema / Pydantic 风格验证、错误反馈与自我修正
# 只使用标准库，模拟 schema 校验和 retry feedback。

import json

invoice_schema = {
    "type": "object",
    "required": ["invoice_id", "total", "currency"],
    "properties": {
        "invoice_id": {"type": ["string", "null"]},
        "total": {"type": ["number", "null"], "minimum": 0},
        "currency": {"type": "string", "enum": ["USD", "CAD", "EUR", "unknown"]},
    },
}


def validate_invoice(data: dict) -> list[str]:
    errors = []
    required = invoice_schema["required"]
    properties = invoice_schema["properties"]

    for field in required:
        if field not in data:
            errors.append(f"缺少必填字段：{field}")

    invoice_id = data.get("invoice_id")
    if invoice_id is not None and not isinstance(invoice_id, str):
        errors.append("invoice_id 必须是 string 或 null")

    total = data.get("total")
    if total is not None:
        if not isinstance(total, (int, float)) or isinstance(total, bool):
            errors.append("total 必须是 number 或 null")
        elif total < properties["total"]["minimum"]:
            errors.append("total 不能为负数")

    if data.get("currency") not in properties["currency"]["enum"]:
        errors.append("currency 必须是 USD、CAD、EUR 或 unknown")

    return errors


def self_correct(candidate: dict, errors: list[str]) -> dict:
    # 真实系统会把 errors、原文和失败 JSON 一起反馈给模型；这里用确定性逻辑模拟修正。
    corrected = dict(candidate)
    if "total 必须是 number 或 null" in errors and isinstance(corrected.get("total"), str):
        corrected["total"] = float(corrected["total"])
    if "currency 必须是 USD、CAD、EUR 或 unknown" in errors:
        corrected["currency"] = "unknown"
    return corrected

candidate = {"invoice_id": "INV-9", "total": "88.10", "currency": "dollars"}
errors = validate_invoice(candidate)
print("第一次验证错误：", errors)
retry_feedback = {"failed_output": candidate, "validation_errors": errors}
print("反馈给模型的重试信息：")
print(json.dumps(retry_feedback, ensure_ascii=False, indent=2))

corrected = self_correct(candidate, errors)
print("修正后输出：", corrected)
print("第二次验证错误：", validate_invoice(corrected))

### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 使用 JSON Schema、工具 schema 或 Pydantic 风格模型定义字段、类型、枚举和范围。
- 对源资料可能缺失的字段允许 `null` 或 `unknown`，避免 required 字段诱导幻觉。
- 验证失败时反馈具体错误，例如字段缺失、类型错误、枚举非法、金额为负。
- 区分语法验证、schema 验证和业务语义验证；三者不能互相替代。

**✗ 反模式**

- 只在 prompt 中写“必须输出 JSON”，没有程序化验证。
- 把所有字段设为不可为空，迫使模型猜测不存在的信息。
- 看到 `json.loads()` 成功就认为答案可信。
- 重试时只说“修正格式”，没有提供具体校验错误。

### 🧪 模拟题

**Q1.** 模型输出 `{"total":"88.10","currency":"dollars"}`，但 schema 要求 `total` 为 number、`currency` 为 `USD|CAD|EUR|unknown`。最合适的下一步是？

A) 把字段级验证错误反馈给模型，要求仅修正无效字段  
B) 接受结果，因为它是合法 JSON  
C) 删除 enum 限制，避免失败  
D) 无限重试，直到模型随机输出正确

> **答案：A**  
> **解析：** 合法 JSON 不等于 schema 合法；具体错误反馈能支持可控 self-correction。

**Q2.** 源文档没有出现供应商税号，但输出 schema 有 `supplier_tax_id`。哪种 schema 设计最稳健？

A) `supplier_tax_id` 可为 `string|null`，并在规则中说明缺失时返回 `null`  
B) `supplier_tax_id` 必须是非空字符串  
C) 要求模型根据公司名推测税号  
D) 把税号字段从验证中移除，但仍要求模型输出

> **答案：A**  
> **解析：** 可空字段把“不知道”表示为合法状态，减少为了满足 required 而编造值的风险。

## D4.3 — 设计验证重试循环提升输出可靠性

### 📖 概念解释

验证重试循环把概率性生成包在确定性控制流中：生成候选结果、验证、把具体错误反馈给模型、再次生成；达到最大重试次数后，返回 fallback、`null` 或升级人工。CCA-F 关注的是“有边界的可靠性”：重试适合修复格式、类型、枚举等可纠正错误，但不能凭空产生源资料中不存在的信息。

良好的 retry loop 必须有最大阈值、可观察的错误记录和明确的升级路径。重试 prompt 不应只说“再试一次”，而应包含原始资料、失败输出、schema/业务错误和期望的修正范围。

In [ ]:
# Task 4.3：验证重试循环、最大阈值与 fallback 升级
# 重点：错误要具体、重试要有上限、失败后要可恢复。

from dataclasses import dataclass

@dataclass
class ExtractionAttempt:
    attempt: int
    output: dict
    errors: list[str]


def simulated_model(attempt: int, feedback: dict | None) -> dict:
    # 第一次犯类型错误；收到具体反馈后修正。真实系统会调用模型 API。
    if attempt == 1:
        return {"invoice_id": "INV-9", "total": "88.10", "currency": "USD"}
    if feedback and "total 必须是 number 或 null" in feedback.get("validation_errors", []):
        return {"invoice_id": "INV-9", "total": 88.10, "currency": "USD"}
    return {"invoice_id": "INV-9", "total": None, "currency": "unknown"}


def validate_extraction(data: dict) -> list[str]:
    errors = []
    if not isinstance(data.get("invoice_id"), str):
        errors.append("invoice_id 必须是 string")
    if data.get("total") is not None and not isinstance(data.get("total"), (int, float)):
        errors.append("total 必须是 number 或 null")
    if data.get("currency") not in {"USD", "CAD", "EUR", "unknown"}:
        errors.append("currency 枚举非法")
    return errors


def extract_with_retries(max_retries: int = 2) -> dict:
    feedback = None
    history: list[ExtractionAttempt] = []

    for attempt in range(1, max_retries + 2):
        output = simulated_model(attempt, feedback)
        errors = validate_extraction(output)
        history.append(ExtractionAttempt(attempt, output, errors))
        print(f"attempt={attempt}", output, "errors=", errors)

        if not errors:
            return {"status": "accepted", "result": output, "attempts": attempt}

        feedback = {"failed_output": output, "validation_errors": errors}

    # 达到阈值后停止，交给 fallback，而不是无限重试。
    return {
        "status": "needs_human_review",
        "result": None,
        "last_errors": history[-1].errors,
        "attempts": len(history),
    }

final_result = extract_with_retries(max_retries=2)
print("final:", final_result)

### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 每次重试都携带原文、失败输出和具体校验错误。
- 设置最大重试次数，并记录每次失败原因，便于审计和调试。
- 对可修复错误重试；对源资料缺失、权限不足或业务风险高的情况升级或返回 `null`。
- fallback 可以是人工审核、保守默认值、排队重处理或让上游重新提供资料。

**✗ 反模式**

- 无限重试，导致延迟、成本和不确定性失控。
- 重试 prompt 只写“再试一次”，没有错误定位。
- 把验证失败都归因于 prompt，不区分 schema 错误、业务错误和资料缺失。
- 达到阈值后静默返回部分错误结果。

### 🧪 模拟题

**Q1.** 一个抽取流程连续两次失败：第一次 `total` 是字符串，第二次 `currency` 不在枚举中。最好的 retry loop 设计是？

A) 每次把具体校验错误、失败输出和原文反馈给模型，并在达到最大次数后升级  
B) 不记录错误，只重复调用模型直到成功  
C) 删除验证器，让流程继续  
D) 只提高 max tokens

> **答案：A**  
> **解析：** 有边界的重试需要具体反馈和最大阈值；失败后要进入明确 fallback。

**Q2.** 达到最大重试次数后仍无法从文档中找到必需证据。哪种处理最符合 CCA-F？

A) 返回 `needs_human_review` 或 `null`，并保留最后的验证错误  
B) 让模型根据常识补全字段  
C) 继续无限重试，因为下一次可能成功  
D) 把错误吞掉，返回最后一次模型输出

> **答案：A**  
> **解析：** 缺失资料不是靠重试解决的问题；可靠系统需要停止条件、fallback 和可观察错误。